## Install Packages for the Environment via Anaconda

Follow these steps to install new packages:

1. **Activate the Environment**  
   After creating the environment, activate it using:  
   **`conda activate ErSE222`**

2. **Install openpyxl**  
   Once the environment is activated, install openpyxl with:  
   **`pip install pandas`**  
   **`pip install openpyxl`** 
   *Usage*: Provides tools for dealing with excels.






## Download the data from Google Drive [Click Here](https://docs.google.com/spreadsheets/d/1kFdt4y0CAZ-e5BtIpREZHydtmg7_FD-n/edit?usp=drive_link&ouid=110267031826258682527&rtpof=true&sd=true)

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

%matplotlib inline

np.random.seed(1)

# Little boilerplate code to find out if we have a gpu
device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")
print(f'Device: {device}')

Cuda installed! Running on GPU!
Device: cuda


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Filename for the Excel file containing well log data
filename = 'data/WellLog/well4tutorial2.xlsx'

# Loop through the specified filename(s)
for fname in [filename]:
    # Read the Excel file into a DataFrame
    df = pd.read_excel(fname)
    
    # Select relevant columns for well log parameters
    df_raw_input = df[['DTCO', 'ECGR', 'RHOB', 'PHIT']]
    
    # Prepare the input features (X) and output labels (y)
    X = df_raw_input[0:]  # Include all rows (Adjust if not divisible)
    y = df['DTSM'][0:]     # Include all rows for DTSM (Adjust if not divisible)
    
    # Convert DataFrame to NumPy arrays
    X = np.array(X)
    y = np.array(y)
    
    # Print the shapes of the input features and output labels
    print(X.shape, y.shape)
    
    # Reshape the input features into a 3D array (shape: [8, 50000, 4])
    Xall = np.reshape(X, [8, 50000, 4], order='F').transpose(1, 0, 2)
    
    # Reshape the output labels into a 2D array (shape: [8, 50000])
    yall = np.reshape(y, [8, 50000], order='F').transpose(1, 0)
    
    # Print the shapes of reshaped input features and output labels
    print(Xall.shape, yall.shape)

# Abbreviations for well log parameters
# DTCO: Compressional Sonic Log
# ECGR: Electrical Conductivity
# RHOB: Bulk Density Log
# PHIT: Total Porosity
# DTSM: Distributed Temperature Sensing Measurement

(400000, 4) (400000,)
(50000, 8, 4) (50000, 8)


## Training Process

## Data preparation

In [31]:
from torch.utils.data import Dataset, DataLoader

# 1. Define your Dataset class (if you haven't already):

class MyDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)  # Convert to torch tensor
        self.labels = torch.tensor(labels, dtype=torch.float32) #Convert labels to long for classification

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [32]:
from sklearn.model_selection import train_test_split

batch_size = 16
# data and labels to tensor
datainput = torch.tensor(Xall,dtype=torch.float32)
labelout = torch.tensor(yall,dtype=torch.float32)


# Divide the data into train and test
train_data, test_data, train_labels, test_labels = train_test_split(
    datainput, labelout, test_size=0.1, random_state=42)

# Create your Dataset instance
dataset_train = MyDataset(train_data, train_labels)
dataset_test  = MyDataset(test_data, test_labels)

# Create a DataLoader
# Adjust batch_size as needed; shuffle=True for training

train_loader = torch.utils.data.DataLoader(dataset=dataset_train, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=dataset_test, 
                                          batch_size=batch_size, 
                                          shuffle=False)

/tmp/ipykernel_862240/1364125503.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.data = torch.tensor(data, dtype=torch.float32)  # Convert to torch tensor
/tmp/ipykernel_862240/1364125503.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(labels, dtype=torch.float32) #Convert labels to long for classification


## Create Model Class

In [5]:
import torch
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, input_dim, output_dim, D1, D2, D3, D4):
        super(RNNModel, self).__init__()
        self.rnn1 = nn.RNN(input_dim, D1, batch_first=True, bidirectional=False)
        self.rnn2 = nn.RNN(D1 , D2, batch_first=True, bidirectional=False)  # *2 for bidirectional
        self.rnn3 = nn.RNN(D2 , D3, batch_first=True, bidirectional=False)
        self.rnn4 = nn.RNN(D3 , D4, batch_first=True, bidirectional=False)

        self.output_layer = nn.Linear(output_dim*D4, output_dim)  # *2 for bidirectional

    def forward(self, x):
        # Forward pass through RNN layers
        x = self.rnn1(x)[0]
        x = self.rnn2(x)[0]
        x = self.rnn3(x)[0]
        x = self.rnn4(x)[0]
        
        f = torch.flatten(x,start_dim=1)
        # Linear layer
        o = self.output_layer(f)
        
        return o

## Training Process

In [6]:
# Parameters
input_dim = 4 # e.g., number of features in your input data
output_dim = 8
D1 = 8  # Number of units in the first layer
D2 = 2*D1  # Number of units in the second layer
D3 = 2*D2  # Number of units in the third layer
D4 = 2*D3  # Number of units in the fourth layer

# Create instances of each model
rnn_model = RNNModel(input_dim=input_dim, output_dim=output_dim, D1=D1, D2=D2, D3=D3, D4=D4).to(device)

print(rnn_model)

RNNModel(
  (rnn1): RNN(4, 8, batch_first=True)
  (rnn2): RNN(8, 16, batch_first=True)
  (rnn3): RNN(16, 32, batch_first=True)
  (rnn4): RNN(32, 64, batch_first=True)
  (output_layer): Linear(in_features=512, out_features=8, bias=True)
)


In [7]:
criterion = nn.MSELoss()
learning_rate = 1e-4
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)  

# Train the model
rnn_model.train()
num_epochs = 50

for epoch in range(num_epochs):
    losall = 0
    for i, (images, labels) in enumerate(train_loader):
        # Load images with gradient accumulation capabilities
        images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
        labels = labels.view(labels.shape[0],labels.shape[1]).to(device)

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()
        
        # Forward pass to get output/logits
        outputs = rnn_model(images)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        losall+= loss.item()
        
        # Getting gradients w.r.t. parameters
        loss.backward()
        
        # Updating parameters
        optimizer.step()
        
    
            
    # Print Loss
    print('epoch: {}. Loss: {}'.format(epoch, losall))
    
# Save Model
torch.save(rnn_model.state_dict(), 'data/WellLog//model_weights_RNN.pth')


epoch: 0. Loss: 26692808.608520508
epoch: 1. Loss: 4129495.0241088867
epoch: 2. Loss: 3594379.8028259277
epoch: 3. Loss: 3594100.4125671387
epoch: 4. Loss: 3589356.418823242
epoch: 5. Loss: 3585801.724304199
epoch: 6. Loss: 3555764.363342285
epoch: 7. Loss: 3513733.333969116
epoch: 8. Loss: 2737753.662864685
epoch: 9. Loss: 2154317.521179199
epoch: 10. Loss: 1991076.6569366455
epoch: 11. Loss: 1883030.9125518799
epoch: 12. Loss: 1782579.7171325684
epoch: 13. Loss: 1710216.1104354858
epoch: 14. Loss: 1657984.3736724854
epoch: 15. Loss: 1618958.6296691895
epoch: 16. Loss: 1577898.0206375122
epoch: 17. Loss: 1544837.6331863403
epoch: 18. Loss: 1519901.521446228
epoch: 19. Loss: 1493924.021659851
epoch: 20. Loss: 1474000.6770706177
epoch: 21. Loss: 1450646.6166000366
epoch: 22. Loss: 1427560.3013076782
epoch: 23. Loss: 1404492.2223472595
epoch: 24. Loss: 1381045.9009895325
epoch: 25. Loss: 1363509.9078483582
epoch: 26. Loss: 1341275.5030899048
epoch: 27. Loss: 1326796.5405540466
epoch: 28.

## Inference (Testing)

In [8]:
# Load Best Model
rnn_model.load_state_dict(torch.load('data/WellLog/model_weights_RNN.pth'))

laball = []
dat = []
outputsall= []
Eall = []
# Iterate through test dataset
for images, labels in test_loader:
    images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
    labels = labels.view(labels.shape[0],labels.shape[1]).to(device)


    # Forward pass to get output/logits
    outputs = rnn_model(images)

    
    laball.append(labels.cpu().numpy())
    dat.append(images.detach().cpu().numpy())
    outputsall.append(outputs.detach().cpu().numpy())

    

outputsall = np.concatenate(outputsall)
laball = np.concatenate(laball)
dat = np.concatenate(dat)

# Print Loss
print('Loss: {}'.format(loss.item()))

Loss: 151.28738403320312


In [9]:
from sklearn.metrics import r2_score, mean_squared_error
y_test = laball
y_pred = outputsall
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

r2, rmse

(0.7036625742912292, 362.5438537597656)

## GRU

In [10]:

# GRU Model
class GRUModel(nn.Module):
    def __init__(self, input_dim, output_dim, D1, D2, D3, D4):
        super(GRUModel, self).__init__()
        self.gru1 = nn.GRU(input_dim, D1, batch_first=True, bidirectional=False)
        self.gru2 = nn.GRU(D1, D2, batch_first=True, bidirectional=False)
        self.gru3 = nn.GRU(D2, D3, batch_first=True, bidirectional=False)
        self.gru4 = nn.GRU(D3, D4, batch_first=True, bidirectional=False)

        self.output_layer = nn.Linear(output_dim * D4, output_dim)  # Adjust output_layer based on requirements

    def forward(self, x):
        # Forward pass through GRU layers
        x, _ = self.gru1(x)
        x, _ = self.gru2(x)
        x, _ = self.gru3(x)
        x, _ = self.gru4(x)

        f = torch.flatten(x, start_dim=1)
        # Linear layer
        o = self.output_layer(f)

        return o

In [11]:
# Parameters
input_dim = 4 # e.g., number of features in your input data
output_dim = 8
D1 = 8  # Number of units in the first layer
D2 = 2*D1  # Number of units in the second layer
D3 = 2*D2  # Number of units in the third layer
D4 = 2*D3  # Number of units in the fourth layer

# Create instances of each model
GRU_model = GRUModel(input_dim=input_dim, output_dim=output_dim, D1=D1, D2=D2, D3=D3, D4=D4).to(device)

print(GRU_model) 

GRUModel(
  (gru1): GRU(4, 8, batch_first=True)
  (gru2): GRU(8, 16, batch_first=True)
  (gru3): GRU(16, 32, batch_first=True)
  (gru4): GRU(32, 64, batch_first=True)
  (output_layer): Linear(in_features=512, out_features=8, bias=True)
)


In [12]:
criterion = nn.MSELoss()
learning_rate = 1e-3
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)  

# Train the model
rnn_model.train()
num_epochs = 50

for epoch in range(num_epochs):
    losall = 0
    for i, (images, labels) in enumerate(train_loader):
        # Load images with gradient accumulation capabilities
        images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
        labels = labels.view(labels.shape[0],labels.shape[1]).to(device)

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()
        
        # Forward pass to get output/logits
        outputs = GRU_model(images)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        losall+= loss.item()
        
        # Getting gradients w.r.t. parameters
        loss.backward()
        
        # Updating parameters
        optimizer.step()
        
    
            
    # Print Loss
    print('epoch: {}. Loss: {}'.format(epoch, losall))
    
# Save Model
torch.save(GRU_model.state_dict(), 'data/WellLog//model_weights_GRU.pth')


epoch: 0. Loss: 70332735.27539062
epoch: 1. Loss: 70338683.04101562
epoch: 2. Loss: 70332665.1171875
epoch: 3. Loss: 70338240.40234375
epoch: 4. Loss: 70333428.21679688
epoch: 5. Loss: 70331787.04882812
epoch: 6. Loss: 70333672.734375
epoch: 7. Loss: 70339392.828125
epoch: 8. Loss: 70331921.78710938
epoch: 9. Loss: 70340213.9765625
epoch: 10. Loss: 70334820.30664062
epoch: 11. Loss: 70333294.76953125
epoch: 12. Loss: 70334049.28320312
epoch: 13. Loss: 70337362.9140625
epoch: 14. Loss: 70331164.48632812
epoch: 15. Loss: 70337674.49804688
epoch: 16. Loss: 70334578.1796875
epoch: 17. Loss: 70336783.10742188
epoch: 18. Loss: 70334607.39453125
epoch: 19. Loss: 70334677.54101562
epoch: 20. Loss: 70337155.25195312
epoch: 21. Loss: 70334808.66796875
epoch: 22. Loss: 70336976.44335938
epoch: 23. Loss: 70335356.3359375
epoch: 24. Loss: 70336172.42382812
epoch: 25. Loss: 70337458.5
epoch: 26. Loss: 70334908.94140625
epoch: 27. Loss: 70332280.07421875
epoch: 28. Loss: 70332847.3125
epoch: 29. Loss

In [26]:
# Load Best Model
GRU_model.load_state_dict(torch.load('data/WellLog/model_weights_GRU.pth'))

laball = []
dat = []
outputsall= []
Eall = []
# Iterate through test dataset
for images, labels in test_loader:
    images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
    labels = labels.view(labels.shape[0],labels.shape[1]).to(device)


    # Forward pass to get output/logits
    outputs = GRU_model(images)

    
    laball.append(labels.cpu().numpy())
    dat.append(images.detach().cpu().numpy())
    outputsall.append(outputs.detach().cpu().numpy())

    

outputsall = np.concatenate(outputsall)
laball = np.concatenate(laball)
dat = np.concatenate(dat)

# Print Loss
print('Loss: {}'.format(loss.item()))

Loss: 87.93310546875


In [27]:
from sklearn.metrics import r2_score, mean_squared_error
y_test = laball
y_pred = outputsall
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

r2, rmse

(-19.277511596679688, 24819.73828125)

## LSTM

In [15]:

# LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_dim, output_dim, D1, D2, D3, D4):
        super(LSTMModel, self).__init__()
        self.lstm1 = nn.LSTM(input_dim, D1, batch_first=True, bidirectional=False)
        self.lstm2 = nn.LSTM(D1, D2, batch_first=True, bidirectional=False)
        self.lstm3 = nn.LSTM(D2, D3, batch_first=True, bidirectional=False)
        self.lstm4 = nn.LSTM(D3, D4, batch_first=True, bidirectional=False)

        self.output_layer = nn.Linear(output_dim * D4, output_dim)  # Adjust output_layer based on requirements

    def forward(self, x):
        # Forward pass through LSTM layers
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x, _ = self.lstm3(x)
        x, _ = self.lstm4(x)

        f = torch.flatten(x, start_dim=1)
        # Linear layer
        o = self.output_layer(f)

        return o

In [16]:
# Parameters
input_dim = 4 # e.g., number of features in your input data
output_dim = 8
D1 = 8  # Number of units in the first layer
D2 = 2*D1  # Number of units in the second layer
D3 = 2*D2  # Number of units in the third layer
D4 = 2*D3  # Number of units in the fourth layer

# Create instances of each model
LSTM_model = LSTMModel(input_dim=input_dim, output_dim=output_dim, D1=D1, D2=D2, D3=D3, D4=D4).to(device)

print(LSTM_model)

LSTMModel(
  (lstm1): LSTM(4, 8, batch_first=True)
  (lstm2): LSTM(8, 16, batch_first=True)
  (lstm3): LSTM(16, 32, batch_first=True)
  (lstm4): LSTM(32, 64, batch_first=True)
  (output_layer): Linear(in_features=512, out_features=8, bias=True)
)


In [17]:
criterion = nn.MSELoss()
learning_rate = 1e-3
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)  

# Train the model
rnn_model.train()
num_epochs = 50

for epoch in range(num_epochs):
    losall = 0
    for i, (images, labels) in enumerate(train_loader):
        # Load images with gradient accumulation capabilities
        images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
        labels = labels.view(labels.shape[0],labels.shape[1]).to(device)

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()
        
        # Forward pass to get output/logits
        outputs = LSTM_model(images)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        losall+= loss.item()
        
        # Getting gradients w.r.t. parameters
        loss.backward()
        
        # Updating parameters
        optimizer.step()
        
    
            
    # Print Loss
    print('epoch: {}. Loss: {}'.format(epoch, losall))
    
# Save Model
torch.save(LSTM_model.state_dict(), 'data/WellLog/model_weights_LSTM.pth')


epoch: 0. Loss: 70331674.72851562
epoch: 1. Loss: 70333051.41601562
epoch: 2. Loss: 70333032.2109375
epoch: 3. Loss: 70332066.54296875
epoch: 4. Loss: 70330319.21289062
epoch: 5. Loss: 70330358.08203125
epoch: 6. Loss: 70336485.359375
epoch: 7. Loss: 70331337.48046875
epoch: 8. Loss: 70330685.3984375
epoch: 9. Loss: 70332383.0
epoch: 10. Loss: 70331984.78320312
epoch: 11. Loss: 70332357.3125
epoch: 12. Loss: 70330361.57617188
epoch: 13. Loss: 70332396.43554688
epoch: 14. Loss: 70337010.91601562
epoch: 15. Loss: 70331688.05273438
epoch: 16. Loss: 70337429.55664062
epoch: 17. Loss: 70332974.94140625
epoch: 18. Loss: 70331624.80664062
epoch: 19. Loss: 70333091.953125
epoch: 20. Loss: 70330764.50976562
epoch: 21. Loss: 70330537.390625
epoch: 22. Loss: 70333119.8984375
epoch: 23. Loss: 70331828.29296875
epoch: 24. Loss: 70332834.046875
epoch: 25. Loss: 70337462.88671875
epoch: 26. Loss: 70329803.98632812
epoch: 27. Loss: 70331593.703125
epoch: 28. Loss: 70330369.4765625
epoch: 29. Loss: 703

In [18]:
# Load Best Model
LSTM_model.load_state_dict(torch.load('data/WellLog/model_weights_LSTM.pth'))

laball = []
dat = []
outputsall= []
Eall = []
# Iterate through test dataset
for images, labels in test_loader:
    images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
    labels = labels.view(labels.shape[0],labels.shape[1]).to(device)


    # Forward pass to get output/logits
    outputs = LSTM_model(images)

    
    laball.append(labels.cpu().numpy())
    dat.append(images.detach().cpu().numpy())
    outputsall.append(outputs.detach().cpu().numpy())

    

outputsall = np.concatenate(outputsall)
laball = np.concatenate(laball)
dat = np.concatenate(dat)

# Print Loss
print('Loss: {}'.format(loss.item()))

Loss: 25596.2421875


In [19]:
from sklearn.metrics import r2_score, mean_squared_error
y_test = laball
y_pred = outputsall
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

r2, rmse

(-19.2769832611084, 24819.109375)

## BI-LSTM

In [20]:

# LSTM Model
class BILSTMModel(nn.Module):
    def __init__(self, input_dim, output_dim, D1, D2, D3, D4):
        super(BILSTMModel, self).__init__()
        self.lstm1 = nn.LSTM(input_dim, D1, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(2*D1, D2, batch_first=True, bidirectional=True)
        self.lstm3 = nn.LSTM(2*D2, D3, batch_first=True, bidirectional=True)
        self.lstm4 = nn.LSTM(2*D3, D4, batch_first=True, bidirectional=True)

        self.output_layer = nn.Linear(output_dim * 2* D4, output_dim)  # Adjust output_layer based on requirements

    def forward(self, x):
        # Forward pass through LSTM layers
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x, _ = self.lstm3(x)
        x, _ = self.lstm4(x)

        f = torch.flatten(x, start_dim=1)
        # Linear layer
        o = self.output_layer(f)

        return o

In [21]:
# Parameters
input_dim = 4 # e.g., number of features in your input data
output_dim = 8
D1 = 8  # Number of units in the first layer
D2 = 2*D1  # Number of units in the second layer
D3 = 2*D2  # Number of units in the third layer
D4 = 2*D3  # Number of units in the fourth layer

# Create instances of each model
BILSTM_model = BILSTMModel(input_dim=input_dim, output_dim=output_dim, D1=D1, D2=D2, D3=D3, D4=D4).to(device)

print(BILSTM_model)

BILSTMModel(
  (lstm1): LSTM(4, 8, batch_first=True, bidirectional=True)
  (lstm2): LSTM(16, 16, batch_first=True, bidirectional=True)
  (lstm3): LSTM(32, 32, batch_first=True, bidirectional=True)
  (lstm4): LSTM(64, 64, batch_first=True, bidirectional=True)
  (output_layer): Linear(in_features=1024, out_features=8, bias=True)
)


In [22]:
criterion = nn.MSELoss()
learning_rate = 1e-3
optimizer = torch.optim.Adam(BILSTM_model.parameters(), lr=learning_rate)  

# Train the model
rnn_model.train()
num_epochs = 50

for epoch in range(num_epochs):
    losall = 0
    for i, (images, labels) in enumerate(train_loader):
        # Load images with gradient accumulation capabilities
        images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
        labels = labels.view(labels.shape[0],labels.shape[1]).to(device)

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()
        
        # Forward pass to get output/logits
        outputs = BILSTM_model(images)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        losall+= loss.item()
        
        # Getting gradients w.r.t. parameters
        loss.backward()
        
        # Updating parameters
        optimizer.step()
        
    
            
    # Print Loss
    print('epoch: {}. Loss: {}'.format(epoch, losall))
    
# Save Model
torch.save(LSTM_model.state_dict(), 'data/WellLog/model_weights_BILSTM.pth')


epoch: 0. Loss: 4108085.9739456177
epoch: 1. Loss: 1528546.5396575928
epoch: 2. Loss: 1384634.672744751
epoch: 3. Loss: 1309502.0837860107
epoch: 4. Loss: 1266819.291004181
epoch: 5. Loss: 1193962.6362953186
epoch: 6. Loss: 1140667.8887825012
epoch: 7. Loss: 1056226.4400177002
epoch: 8. Loss: 983317.0849933624
epoch: 9. Loss: 934874.3310680389
epoch: 10. Loss: 892842.1089324951
epoch: 11. Loss: 878673.8683319092
epoch: 12. Loss: 859456.5783557892
epoch: 13. Loss: 852148.728269577
epoch: 14. Loss: 827843.2219991684
epoch: 15. Loss: 822882.6052713394
epoch: 16. Loss: 803022.8226556778
epoch: 17. Loss: 778633.9548606873
epoch: 18. Loss: 769702.5243082047
epoch: 19. Loss: 750316.2902565002
epoch: 20. Loss: 733098.9463834763
epoch: 21. Loss: 744414.6425800323
epoch: 22. Loss: 717534.6273012161
epoch: 23. Loss: 711897.0660562515
epoch: 24. Loss: 691064.9313850403
epoch: 25. Loss: 680494.5829372406
epoch: 26. Loss: 680677.0328187943
epoch: 27. Loss: 657816.5138015747
epoch: 28. Loss: 644116.4

In [23]:
# Load Best Model
LSTM_model.load_state_dict(torch.load('data/WellLog/model_weights_BILSTM.pth'))

laball = []
dat = []
outputsall= []
Eall = []
# Iterate through test dataset
for images, labels in test_loader:
    images = images.view(images.shape[0], images.shape[1],images.shape[2]).requires_grad_(True).to(device)
    labels = labels.view(labels.shape[0],labels.shape[1]).to(device)


    # Forward pass to get output/logits
    outputs = LSTM_model(images)

    
    laball.append(labels.cpu().numpy())
    dat.append(images.detach().cpu().numpy())
    outputsall.append(outputs.detach().cpu().numpy())

    

outputsall = np.concatenate(outputsall)
laball = np.concatenate(laball)
dat = np.concatenate(dat)

# Print Loss
print('Loss: {}'.format(loss.item()))

Loss: 87.93310546875


In [25]:
from sklearn.metrics import r2_score, mean_squared_error
y_test = laball
y_pred = outputsall
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

r2, rmse

(-19.2769832611084, 24819.109375)